In [2]:
import numpy as np
import pandas as pd

In [3]:
import os
from tensorflow import keras
from keras.preprocessing import image

In [4]:
import cv2

In [ ]:
categories  = ['with_mask', 'without_mask']

In [6]:
data = []
for category in categories:
    path=os.path.join('organized_dataset/train', category)
    label=categories.index(category)
    for file in os.listdir(path):
        img_path=os.path.join(path, file)
        img=cv2.imread(img_path)
        img=cv2.resize(img, (224,224))
        data.append((img, label))

        

In [7]:
len(data)

1279

In [8]:
data

[(array([[[35, 63, 57],
          [28, 55, 46],
          [32, 55, 45],
          ...,
          [26, 38, 35],
          [10, 35, 30],
          [11, 37, 29]],
  
         [[31, 58, 52],
          [29, 53, 45],
          [33, 56, 46],
          ...,
          [48, 60, 56],
          [ 8, 30, 23],
          [18, 41, 33]],
  
         [[28, 53, 48],
          [31, 54, 46],
          [36, 58, 49],
          ...,
          [44, 54, 49],
          [10, 30, 22],
          [28, 48, 38]],
  
         ...,
  
         [[33, 72, 58],
          [36, 72, 59],
          [38, 69, 59],
          ...,
          [11, 13, 16],
          [ 8, 10, 14],
          [ 8,  9, 15]],
  
         [[30, 71, 56],
          [35, 69, 56],
          [37, 66, 55],
          ...,
          [ 8,  9, 13],
          [12, 13, 17],
          [12, 12, 18]],
  
         [[27, 69, 52],
          [31, 65, 51],
          [34, 61, 50],
          ...,
          [12, 13, 17],
          [13, 13, 19],
          [13, 12, 21]]], shape=(

In [9]:
import random
random.shuffle(data)

In [10]:
x=[]
y=[]
for img, label in data:
    x.append(img)
    y.append(label)

In [11]:
x=np.array(x)
y=np.array(y)

In [12]:
x.shape

(1279, 224, 224, 3)

In [13]:
y.shape

(1279,)

In [14]:
x=x/255

In [15]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [16]:
x_train.shape

(1023, 224, 224, 3)

In [17]:
x_test.shape

(256, 224, 224, 3)

In [18]:
from keras.applications.vgg16 import VGG16

In [19]:
vgg=VGG16()

553467096/553467096 ━━━━━━━━━━━━━━━━━━━━ 38s 0us/step


In [20]:
from keras.models import Sequential

In [21]:
model=Sequential()

In [22]:
#take all the layers except the last layer of vgg16 and add it to our model
for layers in vgg.layers[:-1]:
    model.add(layers)

In [23]:
# make the layer non trainabble 
for layer in model.layers:
    layer.trainable=False

In [24]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,260,544 (512.16 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 134,260,544 (512.16 MB)

In [25]:
from keras.layers import Dense, Flatten


In [26]:
model.add(Dense(1, activation='sigmoid'))

In [27]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         4,097 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,264,641 (512.18 MB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 134,260,544 (512.16 MB)

In [28]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [29]:
history = model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 114s 4s/step - accuracy: 0.6452 - loss: 0.6119 - val_accuracy: 0.7891 - val_loss: 0.4611
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 229s 7s/step - accuracy: 0.8827 - loss: 0.3644 - val_accuracy: 0.9141 - val_loss: 0.2908
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 149s 4s/step - accuracy: 0.9218 - loss: 0.2729 - val_accuracy: 0.9180 - val_loss: 0.2367
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 114s 4s/step - accuracy: 0.9404 - loss: 0.2233 - val_accuracy: 0.9297 - val_loss: 0.2012
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 164s 5s/step - accuracy: 0.9492 - loss: 0.1953 - val_accuracy: 0.9414 - val_loss: 0.1871


In [ ]:
# ============================================================
# MODEL EVALUATION: metrics, plots and saved artifacts
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
)

# ------------------------------------------------------------------
# 1. Create a new directory to store all evaluation outputs (csv/png)
# ------------------------------------------------------------------
output_dir = "model_evaluation_results"
os.makedirs(output_dir, exist_ok=True)
print(f"Created (or found existing) output directory: {output_dir}/")

# ------------------------------------------------------------------
# 2. Epoch vs Loss plot -> saved as PNG
# ------------------------------------------------------------------
plt.figure(figsize=(8, 6))
plt.plot(history.history["loss"], marker="o", label="Training Loss")
plt.plot(history.history["val_loss"], marker="o", label="Validation Loss")
plt.title("Epoch vs Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
loss_plot_path = os.path.join(output_dir, "epoch_vs_loss.png")
plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {loss_plot_path}")

# ------------------------------------------------------------------
# 3. Epoch vs Accuracy plot -> saved as PNG
# ------------------------------------------------------------------
plt.figure(figsize=(8, 6))
plt.plot(history.history["accuracy"], marker="o", label="Training Accuracy")
plt.plot(history.history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.title("Epoch vs Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
accuracy_plot_path = os.path.join(output_dir, "epoch_vs_accuracy.png")
plt.savefig(accuracy_plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {accuracy_plot_path}")

# ------------------------------------------------------------------
# 4. Get predictions on the test set (needed for all metrics below)
# ------------------------------------------------------------------
y_pred_prob = model.predict(x_test).ravel()   # predicted probabilities
y_pred = (y_pred_prob > 0.5).astype(int)      # predicted class labels

# ------------------------------------------------------------------
# 5. Accuracy, Precision, Recall, F1
# ------------------------------------------------------------------
test_accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {test_accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# ------------------------------------------------------------------
# 6. Confusion Matrix -> saved as PNG
# ------------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=categories)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
cm_plot_path = os.path.join(output_dir, "confusion_matrix.png")
plt.savefig(cm_plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {cm_plot_path}")

tn, fp, fn, tp = cm.ravel()

# ------------------------------------------------------------------
# 7. ROC Curve and AUC -> saved as PNG
# ------------------------------------------------------------------
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random guess")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True)
roc_plot_path = os.path.join(output_dir, "roc_curve.png")
plt.savefig(roc_plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved: {roc_plot_path}")
print(f"AUC: {roc_auc:.4f}")

# ------------------------------------------------------------------
# 8. Save all metrics (accuracy, precision, recall, confusion matrix
#    values, AUC) into a single CSV file
# ------------------------------------------------------------------
metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy", "Precision", "Recall", "F1 Score", "AUC",
        "True Negative", "False Positive", "False Negative", "True Positive",
    ],
    "Value": [
        test_accuracy, precision, recall, f1, roc_auc,
        tn, fp, fn, tp,
    ],
})
metrics_csv_path = os.path.join(output_dir, "evaluation_metrics.csv")
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"Saved: {metrics_csv_path}")

print("\nAll evaluation plots (PNG) and metrics (CSV) have been saved inside:", output_dir)
metrics_df

In [48]:
model.save('mask_model.keras')
print('Saved model to mask_model.keras')

Saved model to mask_model.keras


In [30]:
cv2.VideoCapture(0)

< cv2.VideoCapture 000002583BBAFC10>

In [31]:
def detect_mask(frame):
    y_pred = model.predict(frame.reshape(1, 224, 224, 3))
    return y_pred[0][0]

In [44]:
def draw_label(frame, text, pos, bg_color):
      text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 1)[0]

      end_x = pos[0] + text_size[0] + 2
      end_y = pos[1] + text_size[1] - 2

      cv2.rectangle(frame, pos, (end_x, end_y), bg_color, cv2.FILLED)
      cv2.putText(frame, text, (pos[0], pos[1] + text_size[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 1)

In [47]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError('Could not open webcam')

frame_count = 0
max_frames = 50

try:
    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        # call the detection method here and get the result
        img = cv2.resize(frame, (224,224))
        coods = detect_face(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
        for (x, y, w, h) in coods:
            cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)

        y_pred = detect_mask(img)
        draw_label(frame, f'Predicted: {y_pred:.2f}', (10, 30), (0,255,0) if y_pred > 0.5 else (0,0,255))
        cv2.imshow('frame', frame)

        frame_count += 1
        if cv2.waitKey(1) & 0xFF == ord('x'):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 

In [ ]:

sample1 = cv2.imread('img.png')
if sample1 is None:
    sample1 = frame.copy()  # fallback to an already available valid image
y_pred = model.predict(sample1.reshape(1, 224, 224, 3))
y_pred = (y_pred > 0.5).astype(int)[0][0]
detect_mask(sample1)

ValueError: cannot reshape array of size 921600 into shape (1,224,224,3)

In [42]:
while True:
    ret,frame=cv2.VideoCapture(0).read()
    coods=detect_face(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    for (x,y,w,h) in coods:
        cv2.rectangle(frame, (x,y), (x+w, y+h), (255,0,0), 2)
        cv2.imshow('frame', frame)

    if cv2.waitKey(1) & 0xFF==ord('x'):
        break

cv2.destroyAllWindows()    

KeyboardInterrupt: 

In [41]:
harr = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
if harr.empty():
    raise FileNotFoundError('Could not load haarcascade_frontalface_default.xml')

In [34]:
def detect_face(img):
    coods=harr.detectMultiScale(img, 1.3, 5)

    return coods